# Convert Student Paper Index spreadsheet to Invenio or DataCite json.

In [ ]:
import pandas as pd
import json
import os

In [ ]:
dir = '/Users/metadatagamechanger/Repositories/MooreaStudentPapers/'
input = dir + 'data/index_to_moorea_class_projects_1992-2022.csv'
input_df = pd.read_csv(input)
input_df.head()

In [ ]:
def rowToMetadata(i: int,           # row index in dataframe
                  papers_df,
                  paperIndex: int,  # index of paper in publication year
                  type:       str   # type of metaata (invenio or dataCite)
                  ):

    if type == 'dataCite':
        metadata_d = {
            "id": "",
            "type": "dois",
            "attributes": {}
        }
        metadata_d['attributes']['creators'] = []              # Add creators to metadata dictionary, initialize as empty list

        for p, person in enumerate(input_df.at[i, 'Authors, Primary'].split(';')):   # Loop the authors and make creators in the metadata dictionary
            person = person.replace('Dr. ','')
            if ',' in str(person):
                toks = person.split(',')
            else:
                toks = person.split(' ,')

            if len(toks) < 2 or len(toks) > 4:
                continue

            family = toks[0].replace(',','') if ',' in str(person) else toks[1]
            family = family.strip()

            given = " ".join(toks[1:]) if ',' in str(person) else toks[0]
            given = given.strip()

            metadata_d['attributes']['creators'].append({
                                                        'nameType': 'personal',
                                                        'given_name':   given,
                                                        'family_name':  family,
                                                        'affiliation': [{
                                                                        "name": "University of California, Berkeley",
                                                                        "affiliationIdentifier": "https://ror.org/01an7q238",
                                                                        "affiliationIdentifierScheme": "ROR"
                                                                        }],
                                                    #    "nameIdentifiers": [{
                                                    #                    "nameIdentifier": ":unkn",
                                                    #                    "nameIdentifierScheme": "ORCID"
                                                    #                    }]
                                                        })

            #
            # Create a unique identifier for each record using a combination of the iPlaces DOI prefix, the abbreviation for the class
            # (Biology and Geology of Tropical Islands), the publication year, the paper index, and creator names.
            # The creator names are concatenated together with underscores, and spaces and special characters are removed or 
            # replaced to ensure the identifier is valid.
            #
            identifier = f"{'_'.join([ x['family_name'].replace(' ','')+'_'+ x['given_name'].replace(' ','').replace('.','') for x in metadata_d['attributes']['creators'] ])}"
            identifier = identifier.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
            identifier = f"10.17913/bgtl/{str(input_df.at[i, 'Pub Year'])}/{paperIndex}/{identifier}"
            metadata_d['id'] = identifier

            if len(str(input_df.at[i, 'Title Primary'])) > 0 and str(input_df.at[i, 'Title Primary']) != 'nan':
                metadata_d['attributes']['title'] = [{"lang": "en", "title": input_df.at[i, 'Title Primary']}]

            metadata_d['attributes']['publicationYear']    = input_df.at[i, 'Pub Year']
            metadata_d['attributes']['publisher']          = {
                                                            "name": "University of California, Berkeley",
                                                            "publisherIdentifier": "https://ror.org/01an7q238",
                                                            "publisherIdentifierScheme": "ROR",
                                                            "lang": "en"
                                                            }
            metadata_d['attributes']['dates'] = [{
                                                "date": str(input_df.at[i, 'Pub Year']),
                                                "dateType": "CopyrightYear"
                                            }],

            metadata_d['attributes']['types'] = [{
                                                  "resourceTypeGeneral": "Preprint",
                                                }]

            contributor_Gump = {
                                "nameType": "organizational",
                                "name": "Gump South Pacific Research Station",
                                "affiliation": [{
                                    "name": "Gump South Pacific Research Station",
                                    "affiliationIdentifier": "https://ror.org/04sk0et52",
                                    "affiliationIdentifierScheme": "ROR",
                                    "lang": "en"
                                }],
                                "contributorType": "Sponsor",
                                "nameIdentifiers": [{
                                        "nameIdentifier": "https://ror.org/04sk0et52",
                                        "nameIdentifierScheme": "ROR"
                                    }]
                                },

            metadata_d['attributes']['contributors']      = [contributor_Gump]

            if len(str(input_df.at[i, 'Keywords'])) > 0 and str(input_df.at[i, 'Keywords']) != 'nan':
                keywords_l = str(input_df.at[i, 'Keywords']).split(';')              # some keywords are delimited with ";"
                if len(keywords_l) == 1:
                    keywords_l = str(input_df.at[i, 'Keywords']).split(',')          # some keywords are delimited with ","

                for subject in keywords_l:
                    subject = subject.strip()
                    if len(subject) > 0:
                        if 'subjects' not in metadata_d['attributes']:
                            metadata_d['attributes']['subjects'] = []
                        metadata_d['attributes']['subjects'].append({'subject': subject})
            #
            # The index spreadsheet includes a column named "organism" that includes fairly high-level organism keywords
            # The metadata includes an element named subjectScheme that identifies keywords of a specific type.
            # In this case that is set to "organism" to differentiate these keywords from the general set
            #  
            if all(['organism' in input_df.columns, 
                    len(str(input_df.at[i, 'organism'])),
                    str(input_df.at[i, 'organism']) != 'nan']):
                if 'subjects' not in metadata_d['attributes']:
                    metadata_d['attributes']['subjects'] = []
                metadata_d['attributes']['subjects'].append({'subject': input_df.at[i, 'organism'], 'subjectScheme':'organism'})

            #
            # The index spreadsheet includes a column named "where" that includes fairly high-level location keywords
            # The metadata includes an element named subjectScheme that identifies keywords of a specific type.
            # In this case that is set to "where" to differentiate these keywords from the general set.
            #  
            if all(['where' in input_df.columns, 
                    len(str(input_df.at[i, 'where'])),
                    str(input_df.at[i, 'where']) != 'nan']):
                if 'geoLocations' not in metadata_d['attributes']:
                    metadata_d['attributes']['geoLocations'] = []
                metadata_d['attributes']['geoLocations'].append({'geoLocationPlace': input_df.at[i, 'where']})

            metadata_d['attributes']['version'] = 1

            metadata_d['attributes']['rightsList'] = [{
                                                            "rights": "Creative Commons Attribution 4.0 International",
                                                            "rightsUri": "https://creativecommons.org/licenses/by/4.0/legalcode",
                                                            "schemeUri": "https://spdx.org/licenses/",
                                                            "rightsIdentifier": "cc-by-4.0",
                                                            "rightsIdentifierScheme": "SPDX"
                                                        }],


        fileName = f"{'_'.join([ x['family_name'].replace(' ','')+'_'+ x['given_name'].replace(' ','').replace('.','') for x in metadata_d['attributes']['creators'] ])}.json"
        fileName = fileName.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
        fileName = f'{str(metadata_d["attributes"]["publicationYear"])}/{paperIndex}_' + fileName

    elif type == 'invenio':
        #    
        # initialize invenio dictionary
        # 
        metadata_d = {
        "access": {
            "files": "public",
            "record": "public"
        },
        "files": {
            "enabled": True
        },
        "metadata": {}
        }
        metadata_d['metadata']['creators'] = []              # Add creators to metadata dictionary, initialize as empty list

        for p, person in enumerate(input_df.at[i, 'Authors, Primary'].split(';')):
            person = person.replace('Dr. ','')
            if ',' in person:
                toks = person.split(',')
            else:
                toks = person.split(' ,')

            if len(toks) < 2 or len(toks) > 4:
                continue

            family = toks[0].replace(',','') if ',' in person else toks[1]
            family = family.strip()

            given = " ".join(toks[1:]) if ',' in person else toks[0]
            given = given.strip()                                       # .replace(' ','+')

            metadata_d['metadata']['creators'].append({'person_or_org': {'family_name': family, 'given_name': given}, 'type': 'personal'})

        if len(str(input_df.at[i, 'Abstract'])) > 0 and str(input_df.at[i, 'Abstract']) != 'nan':
            metadata_d['metadata']['description'] = input_df.at[i, 'Abstract']
        #
        # Create a unique identifier for each record using a combination of the iPlaces DOI prefix, the abbreviation for the class
        # (Biology and Geology of Tropical Islands), the publication year, the paper index, and creator names.
        # The creator names are concatenated together with underscores, and spaces and special characters are removed or 
        # replaced to ensure the identifier is valid.
        #
        identifier = f"{'_'.join([ x['person_or_org']['family_name'].replace(' ','')+'_'+ x['person_or_org']['given_name'].replace(' ','').replace('.','') for x in metadata_d['metadata']['creators'] ])}"
        identifier = identifier.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
        identifier = f"10.17913/bgtl/{str(input_df.at[i, 'Pub Year'])}/{paperIndex}/{identifier}"
        metadata_d['metadata']['identifiers'] = [{'identifier': identifier, 'scheme': 'doi'}]

        metadata_d['metadata']['publication_date']   = input_df.at[i, 'Pub Year']
        metadata_d['metadata']['publisher']          = 'University of California, Berkeley'
        metadata_d['metadata']['resource_type']      = 'publication-preprint'

        contributor_Gump = {                                        # define Gump as a contributor
            "person_or_org": {
                "name": "Gump South Pacific Research Station",
                "type": "organizational",
                "identifiers": [{
                "scheme": "ror",
                "identifier": "04sk0et52"
                }],
            },
            "role": {"id": "Sponsor"},
            "affiliations": [{
                "id": "04sk0et52",
                "name": "Gump South Pacific Research Station",
            }]
        }
        metadata_d['metadata']['contributors']      = [contributor_Gump]

        if len(str(input_df.at[i, 'Keywords'])) > 0 and str(input_df.at[i, 'Keywords']) != 'nan':
            keywords_l = str(input_df.at[i, 'Keywords']).split(';')              # some keywords are delimited with ";"
            if len(keywords_l) == 1:
                keywords_l = str(input_df.at[i, 'Keywords']).split(',')          # some keywords are delimited with ","

            for subject in keywords_l:
                subject = subject.strip()
                if len(subject) > 0:
                    if 'subjects' not in metadata_d['metadata']:
                        metadata_d['metadata']['subjects'] = []
                    metadata_d['metadata']['subjects'].append({'subject': subject})
        #
        # The index spreadsheet includes a column named "organism" that includes fairly high-level organism keywords
        # The metadata includes an element named subjectScheme that identifies keywords of a specific type.
        # In this case that is set to "organism" to differentiate these keywords from the general set
        #  
        if all(['organism' in input_df.columns, 
                len(str(input_df.at[i, 'organism'])),
                str(input_df.at[i, 'organism']) != 'nan']):
            if 'subjects' not in metadata_d['metadata']:
                metadata_d['metadata']['subjects'] = []
            metadata_d['metadata']['subjects'].append({'subject': input_df.at[i, 'organism'], 'subjectScheme':'organism'})
        #
        # The index spreadsheet includes a column named "where" that includes fairly high-level location keywords
        # The metadata includes an element named subjectScheme that identifies keywords of a specific type.
        # In this case that is set to "where" to differentiate these keywords from the general set.
        #  
        if all(['where' in input_df.columns, 
                len(str(input_df.at[i, 'where'])),
                str(input_df.at[i, 'where']) != 'nan']):
            if 'subjects' not in metadata_d['metadata']:
                metadata_d['metadata']['subjects'] = []
            metadata_d['metadata']['subjects'].append({'subject': input_df.at[i, 'where'], 'subjectScheme':'where'})

        if len(str(input_df.at[i, 'Title Primary'])) > 0 and str(input_df.at[i, 'Title Primary']) != 'nan':
            metadata_d['metadata']['title'] = input_df.at[i, 'Title Primary']

        relatedIsPartOf = f"10.17913/bgtl/{str(input_df.at[i, 'Pub Year'])}"
        metadata_d['metadata']['related_identifiers'] = [{'identifier': relatedIsPartOf, 
                                                        'relation_type': {'id': 'ispartof'},
                                                        'resource_type': {'id': 'other'},
                                                        'scheme': 'doi'}]

        metadata_d['metadata']['version'] = 1

        fileName = f"{'_'.join([ x['person_or_org']['family_name'].replace(' ','')+'_'+ x['person_or_org']['given_name'].replace(' ','').replace('.','') for x in metadata_d['metadata']['creators'] ])}.json"
        fileName = fileName.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
        fileName = f'{str(metadata_d["metadata"]["publication_date"])}/{paperIndex}_' + fileName
        
    return metadata_d, fileName


## Create metadata records.
See https://datacite-metadata-schema.readthedocs.io/en/4.7/ 
or https://inveniordm.docs.cern.ch/reference/metadata/ for reference

In [ ]:
currentYear = 0                                             # initialize publication year variable, will be updated for each record in loop below
paperIndex  = 0                                             # initialize paper index variable, will be updated for each record in loop below
fileName    = ""                                            # initialize fileName variable

for i in input_df.index:                                    # loop input rows (one paper/metadata record per row)

    pubYear = input_df.at[i, 'Pub Year']                # update publication year for current record
    if pubYear != currentYear:                          # new year, initialize paperIndex
        paperIndex = 1
        currentYear = pubYear
    else:                                               # same year as previous record, increment paperIndex
        paperIndex += 1

    metadata_d, fileName = rowToMetadata(i, input_df, paperIndex, type='dataCite')     # create metadata dictionary for current record
    output = f'{dir}metadata/{fileName}'
    
    os.makedirs(os.path.dirname(output), exist_ok=True)
    print(f"writing metadata to {'/'.join(output.split('/')[-2:])}")
    with open(output, 'w') as f:
        json.dump(metadata_d, f, indent=2, default=str)
